# 03B 换一种问法，还能找到吗

本次理解句向量、余弦排序和名次融合，并用原文核对三路检索是否找到了直接证据。我们使用真实本地编码器产生的向量，默认读取随课保存的实验结果，避免同学一打开 Notebook 就下载模型。

重新编码的完整代码也在下面；需要安装本节可选依赖并显式打开开关。这里的编码器只负责把文本变成向量，后续生成回答仍统一使用 DeepSeek。它是小型教学用多语言模型，不代表最新或最强的检索模型。


## 学习路线与配套课件

实践 3B：句向量排序与名次融合（`L03-P02`）。

建议先完成本节概念正课，再进入本实践小节。这个 Notebook 可用独立新内核从头运行。先预测，再执行代码、修改一个条件并解释结果。

对应课件稳定编号：L03.05-S01、L03.05-S02、L03.05-S03。页数调整不改变这些编号。

按“问题—代码—观察—练习—可复用结果”的顺序学习。先完成练习，再展开参考分析。回放、保存的真实记录和新的在线请求都会明确标注；在线请求默认关闭。

In [9]:
from pathlib import Path
import sys, json, math, hashlib
from collections import Counter
from dataclasses import dataclass
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "03 资料检索与有依据的回答"
sys.path.insert(0, str(CHAPTER / "代码"))
from retrieval_lab import *
print("本课资料:", CHAPTER.relative_to(ROOT) / "data")


本课资料: 03 资料检索与有依据的回答\data


## 1. 句向量中的每个数字是什么

词频向量的坐标可以对应一个词；神经网络编码得到的稠密向量通常没有这样逐维命名的解释。我们把整个向量看成一种经过训练得到的文本表示，再检查相关句子是否更接近。

本课所用模型输出 384 维句向量。维数本身不保证检索质量。问题与资料必须使用同一个编码器和兼容的编码方式；模型升级后一般需要重建资料向量。

先读保存结果的版本、文本哈希和形状。哈希不一致意味着当前文本与保存编码时的文本不同，应该重跑编码，而不是静默复用旧分数。


In [10]:
import numpy as np
chunks = load_corpus(CHAPTER / "数据")
record = json.loads((CHAPTER / "数据/实验记录/semantic-embeddings.json").read_text(encoding='utf-8'))
queries = record["queries"]
texts = [c.text for c in chunks] + queries
assert [c.id for c in chunks] == record["chunk_ids"]
assert [hashlib.sha256(t.encode()).hexdigest() for t in texts] == record["text_sha256"]
embeddings = np.array(record["embeddings"], dtype=np.float32)
print("来源:", record["kind"], record["model"])
print("版本:", record["revision"])
print("矩阵形状:", embeddings.shape, "；每个输入 -> 一行向量")
print("第一行前 8 维:", embeddings[0, :8])
print("向量长度:", np.linalg.norm(embeddings, axis=1).round(5))
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5)


来源: real_local_encoder sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
版本: e8f8c211226b894fcb81acc59f3b34ba3efd5f42
矩阵形状: (15, 384) ；每个输入 -> 一行向量
第一行前 8 维: [ 0.05896864 -0.01363539  0.00744992 -0.02351545 -0.01547834 -0.02682965
 -0.04620931  0.03087066]
向量长度: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## 2. 从文本实际计算向量

下面是同一编码器的真实调用方式。首次运行会从公开模型仓库下载权重；`revision` 固定版本，`trust_remote_code=False` 禁止通过模型加载器执行仓库提供的自定义模型代码。

先保持开关为 False 完成排序实验；如果希望从头复现，按本节 README 安装 `requirements-semantic.txt` 后打开开关。CPU 即可。编码器有长度上限，超出会截断，所以先切块仍然重要；该上限不是 DeepSeek 的上下文窗口。


In [11]:
RUN_ENCODER = False  # 显式改为 True 才加载/下载模型并重新编码。
if RUN_ENCODER:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer(record["model"], revision=record["revision"],
        device="cpu", cache_folder=str(ROOT / ".local/models"), trust_remote_code=False)
    embeddings = encoder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    print("本次重新编码:", embeddings.shape, "；长度上限:", encoder.max_seq_length)
else:
    print("正在使用随课保存的真实编码结果；没有下载模型或发送 API 请求。")


正在使用随课保存的真实编码结果；没有下载模型或发送 API 请求。


## 3. 单位向量的点积就是余弦

资料矩阵形状是 `[片段数, 384]`，一个问题的向量形状是 `[384]`。矩阵乘向量后，每个片段得到一个分数。我们只取分数最高的 k 段；最高只是相对于当前候选集合，并不保证语义正确。

先分别查看“学习记录保存在什么文件”和“昨天没学完的内容今天怎么接着学”。观察被排在前面的原文，不只看数字。


In [12]:
document_vectors = embeddings[:len(chunks)]
query_vectors = embeddings[len(chunks):]
def semantic_hits(query_number, k=3):
    scores = document_vectors @ query_vectors[query_number]
    order = np.argsort(-scores, kind="stable")[:k]
    return [{"chunk": chunks[int(i)], "score": float(scores[i])} for i in order]
for qn, query in enumerate(queries):
    print("\n问题:", query)
    for hit in semantic_hits(qn):
        print(round(hit["score"],4), hit["chunk"].citation)
        print(hit["chunk"].text)
        print("-------------------")



问题: 学习记录保存在什么文件？
0.6081 study-guide.md:1-3
# StudyBox 使用说明（本课程自编资料）
学习记录保存在 progress.json；每条记录包含学习目标、最近一次练习和下一步计划。
没有完成的练习不会标记为已掌握；学生可以保留尝试记录后继续。
-------------------
0.5047 study-guide.md:3-5
没有完成的练习不会标记为已掌握；学生可以保留尝试记录后继续。

# 配置入口
-------------------
0.3781 unrelated.md:1-3
# 教室值日（本课程自编干扰资料）
小组每周三整理桌面，周五检查投影设备。
图书借阅期限为三十天，延期请联系图书管理员。
-------------------

问题: 昨天没学完的内容今天怎么接着学？
0.6169 faq.md:5-7
问：如何继续昨天的学习？
答：读取 progress.json 中的下一步计划，再检查资料版本是否变化。

-------------------
0.4755 study-guide.md:1-3
# StudyBox 使用说明（本课程自编资料）
学习记录保存在 progress.json；每条记录包含学习目标、最近一次练习和下一步计划。
没有完成的练习不会标记为已掌握；学生可以保留尝试记录后继续。
-------------------
0.4447 faq.md:3-5
答：演示客户端使用 timeout 参数设置等待上限；超时不说明任务一定没有在远端执行。

问：如何继续昨天的学习？
-------------------

问题: 未设置 STUDYBOX_CONFIG 时从哪里找配置？
0.5985 study-guide.md:5-7
# 配置入口
load_settings 函数读取 STUDYBOX_CONFIG 指向的配置文件。
没有设置环境变量时使用相对路径 settings.json；相对路径以进程当前工作目录为起点。
-------------------
0.3781 study-guide.md:1-3
# StudyBox 使用说明（本课程自编资料）
学习记录保存在 progress.json；每条记录包含学习目标、最近一次练习和下一步计划。
没有完成的练

## 4. 对照要固定问题和资料

用同一批片段、同一问题与相同 k，对比关键词检索和语义检索。编码器对同义表达可能更有帮助；关键词对函数名、错误码等精确标识符可能更有优势。这是需要验证的经验判断，不是两种方法必然的高低关系。

现在只比较三个自编问题。样本太少，也没有独立测试集，因此不能据此宣称某个模型在中文仓库检索上整体更好。


In [13]:
index = TfidfIndex(chunks)
for qn, question in enumerate(queries):
    print("\n问题:", question)
    print("关键词:", [h["chunk"].citation for h in index.search(question, 3)])
    print("句向量:", [h["chunk"].citation for h in semantic_hits(qn, 3)])



问题: 学习记录保存在什么文件？
关键词: ['study-guide.md:1-3', 'unrelated.md:3-4', 'study-guide.md:7-9']
句向量: ['study-guide.md:1-3', 'study-guide.md:3-5', 'unrelated.md:1-3']

问题: 昨天没学完的内容今天怎么接着学？
关键词: ['faq.md:5-7', 'faq.md:3-5']
句向量: ['faq.md:5-7', 'study-guide.md:1-3', 'faq.md:3-5']

问题: 未设置 STUDYBOX_CONFIG 时从哪里找配置？
关键词: ['study-guide.md:5-7', 'study-guide.md:7-9', 'study-guide.md:3-5']
句向量: ['study-guide.md:5-7', 'study-guide.md:1-3', 'study-guide.md:3-5']


## 5. 先读两路分歧，再决定是否融合

混合检索不是默认把两种方法都打开。先固定问题、资料和 k，查看两路共同召回了什么、各自又独有了什么。分歧能帮助我们判断：是精确标识符更重要，还是表达变化更常见。

下面生成一份可直接用于检索排错的分歧报告。


In [14]:
diagnostic_query = queries[1]
diagnostic_k = 3
lexical_hits = index.search(diagnostic_query, diagnostic_k)
semantic_results = semantic_hits(1, diagnostic_k)
lexical_ids = [hit["chunk"].id for hit in lexical_hits]
semantic_ids = [hit["chunk"].id for hit in semantic_results]
by_id = {chunk.id: chunk.citation for chunk in chunks}
agreement = [by_id[cid] for cid in lexical_ids if cid in semantic_ids]
lexical_only = [by_id[cid] for cid in lexical_ids if cid not in semantic_ids]
semantic_only = [by_id[cid] for cid in semantic_ids if cid not in lexical_ids]
print("问题:", diagnostic_query)
print("共同召回:", agreement)
print("仅词项检索:", lexical_only)
print("仅语义检索:", semantic_only)
assert set(agreement).isdisjoint(lexical_only)
assert set(agreement).isdisjoint(semantic_only)


问题: 昨天没学完的内容今天怎么接着学？
共同召回: ['faq.md:5-7', 'faq.md:3-5']
仅词项检索: []
仅语义检索: ['study-guide.md:1-3']


**结果怎样进入工程判断**

共同召回说明两路对候选有一致判断；单路独有片段需要回到原文检查，不能因为“只有一路找到”就直接删除。只有真实查询集反复出现互补结果，混合检索的额外成本才有依据。


## 6. 混合检索与名次融合

两个方法的原始分数含义不同，直接相加会受尺度影响。一个简单做法是只融合名次：

$$\mathrm{RRF}(d)=\sum_j\frac{1}{c+\mathrm{rank}_j(d)}$$

j 是某一路检索，名次从 1 开始，c 是避免名次差异过大的常数；本课取 60 作为教学配置。没有在某路出现的片段不从该路获得分数。

混合检索不必然提升结果。下面用真实的两路排序计算融合；然后检查合并后的片段是否重复，是否包含回答所需条件。


In [ ]:
def reciprocal_rank_fusion(rankings, offset=60, limit=3):
    """只融合名次；不同编码器的原始分数通常不能直接相加。"""
    if offset <= 0 or limit < 1:
        raise ValueError("offset 和 limit 必须为正数")
    scores, by_id = Counter(), {}
    for ranking in rankings:
        seen = set()
        for rank, hit in enumerate(ranking, 1):
            chunk = hit['chunk']
            if chunk.id in seen:
                continue
            seen.add(chunk.id)
            by_id[chunk.id] = chunk
            scores[chunk.id] += 1/(offset+rank)
    return [{'chunk': by_id[cid], 'score': score} for cid, score in
            sorted(scores.items(), key=lambda item: (-item[1], item[0]))[:limit]]


In [30]:
lexical = index.search(queries[1], k=5)
dense = semantic_hits(1, k=5)
mixed = reciprocal_rank_fusion([lexical, dense],30,5)
for hit in mixed:
    print(round(hit["score"], 5), hit["chunk"].citation)
# 融合分数也不是概率；它只用于这个候选集合的排序。


0.06452 faq.md:5-7
0.06155 faq.md:3-5
0.03125 study-guide.md:1-3
0.02941 study-guide.md:3-5
0.02857 study-guide.md:9-11


### RRF 融合的是名次，不是原始分数

配套课件：L03.05-S01、L03.05-S02。

设关键词排名是 A、B、C，向量排名是 B、C、A。每份排名给候选贡献 $1/(60+r)$，r 从 1 开始。这样无需把两种不同尺度的原始分数直接相加。候选缺席某份列表时，那一份贡献为 0。

In [34]:
case_lists = [["A", "B", "C"], ["B", "C", "A"]]
case_totals = {}
for case_name in ["A", "B", "C"]:
    case_ranks = [order.index(case_name) + 1 for order in case_lists]
    case_parts = [1 / (60 + rank) for rank in case_ranks]
    case_totals[case_name] = sum(case_parts)
    print(case_name, "名次:", case_ranks, "贡献:", case_parts, "合计:", round(sum(case_parts), 6))
print("融合顺序:", sorted(case_totals, key=case_totals.get, reverse=True))
assert case_totals["B"] > case_totals["A"] > case_totals["C"]
# 使用真实融合函数再验证同一组名次；score 故意设为同值。
case_chunks = {name: chunk_text(name, name+'.txt')[0] for name in case_totals}
case_rankings = [[{"chunk": case_chunks[name], "score": 1.0} for name in order] for order in case_lists]
case_fused = reciprocal_rank_fusion(case_rankings, offset=60, limit=3)
assert [h["chunk"].source for h in case_fused] == ["B.txt", "A.txt", "C.txt"]


A 名次: [1, 3] 贡献: [0.01639344262295082, 0.015873015873015872] 合计: 0.032266
B 名次: [2, 1] 贡献: [0.016129032258064516, 0.01639344262295082] 合计: 0.032522
C 名次: [3, 2] 贡献: [0.015873015873015872, 0.016129032258064516] 合计: 0.032002
融合顺序: ['B', 'A', 'C']


**结果解读**

B 约为 0.032522，A 约为 0.032266，C 约为 0.032002。A 虽有一次第一名，B 的另一份排名更靠前，所以 B 总分更高。融合也无法补回两份候选集都漏掉的证据。

**小练习**

从第二份列表移除 A，用贡献 0 处理缺席。说明候选召回范围怎样影响融合结果。

<details><summary>完成后展开参考分析</summary>

A 只保留第一份的 1/61。应先保证候选范围，再讨论如何融合；分数不是正确性概率。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

In [35]:
case_lists = [["A", "B", "C"], ["B", "C"]]   # 第二份移除了 A
case_totals = {}

for case_name in ["A", "B", "C"]:
    case_parts = []
    for order in case_lists:
        if case_name in order:
            rank = order.index(case_name) + 1
            case_parts.append(1 / (60 + rank))
        else:
            case_parts.append(0.0)          # 缺席某一路，贡献为 0
    case_totals[case_name] = sum(case_parts)
    print(case_name, "贡献:", case_parts, "合计:", round(sum(case_parts), 6))

print("融合顺序:", sorted(case_totals, key=case_totals.get, reverse=True))

A 贡献: [0.01639344262295082, 0.0] 合计: 0.016393
B 贡献: [0.016129032258064516, 0.01639344262295082] 合计: 0.032522
C 贡献: [0.015873015873015872, 0.016129032258064516] 合计: 0.032002
融合顺序: ['B', 'C', 'A']


## 7. 重排的位置

初次检索先找候选，重排再更仔细地比较“问题与候选是否匹配”。例如 Cross-Encoder 把问题和候选一起输入一个打分模型；代价通常比分别预先编码资料更高。本节不再增加另一个模型下载，先做一个**明确标为规则的重排小例子**。

如果用户问某个精确标识符，我们可优先检查确实包含它的候选。本次真实候选的首位已经包含该标识符，所以规则可能完全不改变顺序；这不能被讲成重排带来了提升。下一格用人工核对的证据评估三路排序，并展示选错规则如何把错误片段推到首位。这个规则不等同于 Cross-Encoder。


In [36]:
identifier = "STUDYBOX_CONFIG"
candidates = reciprocal_rank_fusion([index.search(queries[2], 5), semantic_hits(2, 5)], limit=5)
reranked = sorted(candidates, key=lambda h: (identifier not in h["chunk"].text, -h["score"]))
print("重排前:", [h["chunk"].citation for h in candidates])
print("规则重排后:", [h["chunk"].citation for h in reranked])
assert {h["chunk"].id for h in candidates} == {h["chunk"].id for h in reranked}


重排前: ['study-guide.md:5-7', 'study-guide.md:3-5', 'unrelated.md:3-4', 'study-guide.md:1-3', 'study-guide.md:7-9']
规则重排后: ['study-guide.md:5-7', 'study-guide.md:3-5', 'unrelated.md:3-4', 'study-guide.md:1-3', 'study-guide.md:7-9']


### 用人工核对的证据检验三路排序

配套课件：L03.05-S02、L03.05-S03。

先从原文为三个自编问题各标出一段直接支持答案的片段，再对同一问题、同一资料和同一候选上限，比较词项检索、真实句向量与 RRF。Hit@1 只问第一条是否命中，RR@5 记录第一条支持证据的位置；它们衡量这组检索结果，不衡量生成回答正确率。

In [37]:
import pandas as pd
gold_citations = {
    queries[0]: "study-guide.md:1-3",
    queries[1]: "faq.md:5-7",
    queries[2]: "study-guide.md:5-7",
}
audit_rows = []
for qn, question in enumerate(queries):
    lexical = index.search(question, 5)
    dense = semantic_hits(qn, 5)
    rankings = {
        "词项": lexical,
        "句向量": dense,
        "RRF": reciprocal_rank_fusion([lexical, dense], limit=5),
    }
    for method, hits in rankings.items():
        citations = [h["chunk"].citation for h in hits]
        rank = citations.index(gold_citations[question]) + 1 if gold_citations[question] in citations else None
        audit_rows.append({
            "问题编号": qn + 1, "检索器": method, "支持证据": gold_citations[question],
            "首条支持证据名次": rank, "Hit@1": int(rank == 1),
            "Hit@3": int(rank is not None and rank <= 3),
            "RR@5": round(1 / rank, 3) if rank is not None else 0.0,
            "前五条": citations,
        })
audit_table = pd.DataFrame(audit_rows)
print(audit_table[["问题编号", "检索器", "首条支持证据名次", "Hit@1", "Hit@3", "RR@5"]].to_string(index=False))
print("三题 Hit@1 汇总:", audit_table.groupby("检索器", sort=False)["Hit@1"].sum().to_dict())

# 与上一格同一批真实候选：故意使用一个不属于当前问题的标识符。
wrong_identifier = "progress.json"
wrong_reranked = sorted(candidates, key=lambda h: (wrong_identifier not in h["chunk"].text, -h["score"]))
before_top = candidates[0]["chunk"].citation
after_top = wrong_reranked[0]["chunk"].citation
print("不合题意的规则：", wrong_identifier, "；第一条:", before_top, "→", after_top)
assert before_top == gold_citations[queries[2]]
assert after_top != gold_citations[queries[2]]
assert {h["chunk"].id for h in candidates} == {h["chunk"].id for h in wrong_reranked}

audit_dir = ROOT / ".local/student-runs/l03"
audit_dir.mkdir(parents=True, exist_ok=True)
audit_path = audit_dir / "ranking-audit.json"
audit_report = {
    "encoder": {"model": record["model"], "revision": record["revision"]},
    "gold_citations": gold_citations,
    "rows": audit_rows,
    "counterexample": {"rule": wrong_identifier, "before_top": before_top, "after_top": after_top},
}
audit_path.write_text(json.dumps(audit_report, ensure_ascii=False, indent=2), encoding="utf-8")
assert json.loads(audit_path.read_text(encoding="utf-8"))["rows"] == audit_rows
print("可复查报告:", audit_path.relative_to(ROOT))

 问题编号 检索器  首条支持证据名次  Hit@1  Hit@3  RR@5
    1  词项         1      1      1   1.0
    1 句向量         1      1      1   1.0
    1 RRF         1      1      1   1.0
    2  词项         1      1      1   1.0
    2 句向量         1      1      1   1.0
    2 RRF         1      1      1   1.0
    3  词项         1      1      1   1.0
    3 句向量         1      1      1   1.0
    3 RRF         1      1      1   1.0
三题 Hit@1 汇总: {'词项': 3, '句向量': 3, 'RRF': 3}
不合题意的规则： progress.json ；第一条: study-guide.md:5-7 → study-guide.md:1-3
可复查报告: .local\student-runs\l03\ranking-audit.json


**结果解读**

这三道小题里，三路方法的第一条都包含人工标出的直接证据，RRF 没有可宣称的命中率提升。上一格的合题标识符也没有改变首位；改用与当前问题无关的 progress.json 后，规则反而把错误片段提到第一。重排只能换顺序，不能自动保证更好。

**小练习**

把表中的 Hit@1 与 Hit@3 分开读，再指出如果正确片段根本不在前五条，重排还能不能找回它。请为第三题核对人工标注的原文。

<details><summary>完成后展开参考分析</summary>

本例两项指标都为满分，只说明三道小题太容易；不能推断方法在其他问题上相同。候选集合里没有正确片段时，重排无从找回。第三题需要读 study-guide.md 的配置入口，确认未设置环境变量时使用 settings.json。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

### 练习：只调整 k

把关键词与语义检索的 k 从 1 改为 5，再融合。记录候选数、人工标出的支持证据名次和新增的无关片段；如果没有新增有用证据，也请如实写出。为什么“返回更多”不能直接等同于“回答更好”？


In [20]:
student_k = 5  # TODO：再改成 5，逐段读返回文本。
result = reciprocal_rank_fusion([index.search(queries[1], student_k), semantic_hits(1, student_k)], limit=student_k)
for h in result:
    print(h["chunk"].citation, h["chunk"].text)


faq.md:5-7 问：如何继续昨天的学习？
答：读取 progress.json 中的下一步计划，再检查资料版本是否变化。

faq.md:3-5 答：演示客户端使用 timeout 参数设置等待上限；超时不说明任务一定没有在远端执行。

问：如何继续昨天的学习？
study-guide.md:1-3 # StudyBox 使用说明（本课程自编资料）
学习记录保存在 progress.json；每条记录包含学习目标、最近一次练习和下一步计划。
没有完成的练习不会标记为已掌握；学生可以保留尝试记录后继续。
study-guide.md:3-5 没有完成的练习不会标记为已掌握；学生可以保留尝试记录后继续。

# 配置入口
study-guide.md:9-11 
# 学习方式
学习建议是先阅读说明，再运行一个最小案例，最后修改一个条件检查理解。


### 参考分析

k 增大通常扩大候选覆盖，但也增加后续输入、重复和无关信息。这三道教学题的直接证据本来都排第一，不能把增加候选数说成命中率提升。请按上面的审计表核对实际新增片段；如果某段不相关，增加它可能干扰回答。第七节会正式讨论更大的任务集、对照和成功率。

## 本次小结

我们检查了真实句向量的版本与来源，计算余弦排序，再与关键词检索融合，并留下按问题核对的审计报告。三道小题不能证明融合优于单路；错误重排规则还会使首位变差。重排是候选之后的一步，精确匹配规则与学习得到的重排器要分清.

参考：[Sentence Transformers 语义检索](https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)、[模型卡](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)、[Elastic 对 RRF 的定义](https://www.elastic.co/docs/reference/elasticsearch/rest-apis/reciprocal-rank-fusion)。核验日期：2026-09-07。
